In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
# plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

import pandas as pd

path_3b = Path("../events/MG3/dataframes/threeTag_picoAOD.h5")
path_4b = Path("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
path_signal = Path("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b = pd.read_hdf(path_3b)
df_bg4b = pd.read_hdf(path_4b)
df_signal = pd.read_hdf(path_signal)
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]
loaded_df = {path_3b: df_3b, path_4b: df_bg4b, path_signal: df_signal}

In [4]:
TrainingInfo.update_metadata()

2025-03-27 22:34:03,671 - INFO - Adding 0 files, removing 0 hashes
0it [00:00, ?it/s]


In [5]:
metadata = TrainingInfo.load_metadata()
hparams = metadata.values()
experiment_names = [hp["experiment_name"] for hp in hparams]
experiment_names = np.unique(experiment_names)

In [7]:
experiment_names
recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    "base_fvt_training_ensemble",
    "base_fvt_training_ensemble_HH4b_400",
    "base_fvt_training_ensemble_HH4b_800",
    "smeared_fvt_training_ensemble",
    "smeared_fvt_training_ensemble_HH4b_400",
]

for experiment_name in recent_experiment_names:
    hashes = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=False, use_cached_metadata=True)
    print(experiment_name, len(hashes))
    hashes_sorted = sorted(hashes)
    # print(hashes_sorted[:2])
    # print(hashes_sorted[-2:])


CR_fvt_training_ensemble_max_fvt 1000
CR_fvt_training_ensemble_max_fvt_HH4b_400 800
CR_fvt_training_ensemble_max_smeared 6000
CR_fvt_training_ensemble_max_smeared_HH4b_400 3200
base_fvt_training_ensemble 3750
base_fvt_training_ensemble_HH4b_400 3000
base_fvt_training_ensemble_HH4b_800 1539
smeared_fvt_training_ensemble 22500
smeared_fvt_training_ensemble_HH4b_400 12000


# STEP 1 Sanity Check

In [ ]:
for experiment_name in ["base_fvt_training_ensemble", 
                        "base_fvt_training_ensemble_HH4b_400", 
                        "base_fvt_training_ensemble_HH4b_800"]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=True
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        assert train_seed == model_seed == data_seed
        sanity_list.append((sr, seed, train_seed))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list, counts)
    assert np.all(counts == 1)
    if experiment_name == "base_fvt_training_ensemble":
        assert len(unique_list) == 5 * 50 * 15
    elif experiment_name == "base_fvt_training_ensemble_HH4b_400":
        assert len(unique_list) == 4 * 50 * 15
    elif experiment_name == "base_fvt_training_ensemble_HH4b_800":
        assert len(unique_list) == 4 * 50 * 15


base_fvt_training_ensemble
[[0.0e+00 0.0e+00 0.0e+00]
 [0.0e+00 0.0e+00 1.0e+00]
 [0.0e+00 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.2e+01]
 [2.0e-02 4.9e+01 1.3e+01]
 [2.0e-02 4.9e+01 1.4e+01]] [1 1 1 ... 1 1 1]
base_fvt_training_ensemble_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00]
 [5.0e-03 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.2e+01]
 [2.0e-02 4.9e+01 1.3e+01]
 [2.0e-02 4.9e+01 1.4e+01]] [1 1 1 ... 1 1 1]


# STEP 2 Sanity Check

In [9]:

for experiment_name in ["smeared_fvt_training_ensemble", "smeared_fvt_training_ensemble_HH4b_400"]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        noise_scale = hp["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        sanity_list.append((sr, seed, train_seed, noise_scale))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list, counts)
    assert np.all(counts == 1)
    if experiment_name == "smeared_fvt_training_ensemble":
        if len(unique_list) != 5 * 50 * 15 * 6:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 15 * 6}")
            print(len(unique_list))
    elif experiment_name == "smeared_fvt_training_ensemble_HH4b_400":
        if len(unique_list) != 4 * 50 * 15 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 15 * 4}")
            print(len(unique_list))

smeared_fvt_training_ensemble
[[0.0e+00 0.0e+00 0.0e+00 5.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00]
 [0.0e+00 0.0e+00 0.0e+00 1.5e+00]
 ...
 [2.0e-02 4.9e+01 1.4e+01 2.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 2.5e+00]
 [2.0e-02 4.9e+01 1.4e+01 3.0e+00]] [1 1 1 ... 1 1 1]
smeared_fvt_training_ensemble_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 5.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 1.0e+00]
 [5.0e-03 0.0e+00 0.0e+00 2.0e+00]
 ...
 [2.0e-02 4.9e+01 1.4e+01 1.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 2.0e+00]
 [2.0e-02 4.9e+01 1.4e+01 3.0e+00]] [1 1 1 ... 1 1 1]


# STEP 3 Sanity Check

In [10]:
# Sanity check

experiment_names
recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",]

metadata = TrainingInfo.load_metadata()
existing_hashes = metadata.keys()
hashes_to_rerun = []
for experiment_name in recent_experiment_names:
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=True
    )
    for hash_, hparam in zip(hashes, hparams):
        SR_stat_hashes = hparam["signal_region"]["SR_stats_hashes"]
        non_existing_SR_stats_hashes_flag = False
        n_SR_stats_not_15_flag = False
        
        for SR_stat_hash in SR_stat_hashes:
            if SR_stat_hash not in existing_hashes:
                print(f"{hash_}: SR_stat_hash {SR_stat_hash} not in existing_hashes")
                print(experiment_name)
                dataset_hp = hparam["dataset"]
                print("signal_ratio", dataset_hp["signal_ratio"])
                # assert dataset_hp["signal_ratio"] == 0.02
                print("-"*100)
                non_existing_SR_stats_hashes_flag = True
        
        if len(SR_stat_hashes) != 15:
            print(f"{hash_}: has {len(SR_stat_hashes)} SR_stats")
            print(experiment_name)
            dataset_hp = hparam["dataset"]
            print("signal_ratio", dataset_hp["signal_ratio"])
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
                                        "CR_fvt_training_ensemble_max_fvt"]
            print("-"*100)
            n_SR_stats_not_15_flag = True
            
        if non_existing_SR_stats_hashes_flag or n_SR_stats_not_15_flag:
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            # assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
            #                             "CR_fvt_training_ensemble_max_fvt"]
            hashes_to_rerun.append(hash_)


In [11]:
# Sanity check

experiment_names
recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    ]

metadata = TrainingInfo.load_metadata()
existing_hashes = metadata.keys()
hashes_to_rerun = []
for experiment_name in recent_experiment_names:
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
    )
    for hash_, hparam in zip(hashes, hparams):
        SR_stat_hashes = hparam["signal_region"]["SR_stats_hashes"]
        non_existing_SR_stats_hashes_flag = False
        n_SR_stats_not_15_flag = False
        
        for SR_stat_hash in SR_stat_hashes:
            if SR_stat_hash not in existing_hashes:
                print(f"{hash_}: SR_stat_hash {SR_stat_hash} not in existing_hashes")
                print(experiment_name)
                dataset_hp = hparam["dataset"]
                print("signal_ratio", dataset_hp["signal_ratio"])
                # assert dataset_hp["signal_ratio"] == 0.02
                print("-"*100)
                non_existing_SR_stats_hashes_flag = True
        
        if len(SR_stat_hashes) != 15:
            print(f"{hash_}: has {len(SR_stat_hashes)} SR_stats")
            print(experiment_name)
            dataset_hp = hparam["dataset"]
            print("signal_ratio", dataset_hp["signal_ratio"])
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            assert experiment_name in ["CR_fvt_training_ensemble_max_smeared",
                                        "CR_fvt_training_ensemble_max_fvt"]
            print("-"*100)
            n_SR_stats_not_15_flag = True
            
        if non_existing_SR_stats_hashes_flag or n_SR_stats_not_15_flag:
            assert dataset_hp["signal_ratio"] in [0.0, 0.02]
            hashes_to_rerun.append(hash_)


In [12]:
for experiment_name in [
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_fvt"
    ]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        sr_size = hp["signal_region"]["4b_in_SR"]
        SR_stat_hashes = hp["signal_region"]["SR_stats_hashes"]
        noise_scale = metadata[SR_stat_hashes[0]]["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        assert len(SR_stat_hashes) == 15
        sanity_list.append((sr, seed, train_seed, noise_scale, sr_size))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list)
    assert np.all(counts == 1)
    if experiment_name == "CR_fvt_training_ensemble_max_smeared":
        if len(unique_list) != 5 * 50 * 4 * 6:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 4 * 6}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_fvt":
        if len(unique_list) != 5 * 50 * 4:
            print(f"Total number of unique configs is not correct, should be {5 * 50 * 4}")
            print(len(unique_list))


CR_fvt_training_ensemble_max_smeared
[[0.0e+00 0.0e+00 0.0e+00 5.0e-01 5.0e-02]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-01 1.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 5.0e-01 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 2.0e-01]]
CR_fvt_training_ensemble_max_fvt
[[0.0e+00 0.0e+00 0.0e+00 1.0e+00 5.0e-02]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00 1.0e-01]
 [0.0e+00 0.0e+00 0.0e+00 1.0e+00 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 2.0e-01]]


In [13]:
for experiment_name in [
    "CR_fvt_training_ensemble_max_smeared_HH4b_400", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400"
    ]:
    print(experiment_name)
    sanity_list = []
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=False
    )
    for hp in hparams:
        sr = hp["dataset"]["signal_ratio"]
        seed = hp["dataset"]["seed"]
        train_seed = hp["train_seed"]
        model_seed = hp["model_seed"]
        data_seed = hp["data_seed"]
        sr_size = hp["signal_region"]["4b_in_SR"]
        SR_stat_hashes = hp["signal_region"]["SR_stats_hashes"]
        noise_scale = metadata[SR_stat_hashes[0]]["smearing"]["noise_scale"]
        assert train_seed == model_seed == data_seed
        assert len(SR_stat_hashes) == 15
        sanity_list.append((sr, seed, train_seed, noise_scale, sr_size))

    unique_list, counts = np.unique(sanity_list, return_counts=True, axis=0)
    print(unique_list)
    assert np.all(counts == 1)
    if experiment_name == "CR_fvt_training_ensemble_max_smeared_HH4b_400":
        if len(unique_list) != 4 * 50 * 4 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4 * 4}")
            print(len(unique_list))
    elif experiment_name == "CR_fvt_training_ensemble_max_fvt_HH4b_400":
        if len(unique_list) != 4 * 50 * 4:
            print(f"Total number of unique configs is not correct, should be {4 * 50 * 4}")
            print(len(unique_list))


CR_fvt_training_ensemble_max_smeared_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 5.0e-01 5.0e-02]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-01 1.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 5.0e-01 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 3.0e+00 2.0e-01]]
CR_fvt_training_ensemble_max_fvt_HH4b_400
[[5.0e-03 0.0e+00 0.0e+00 1.0e+00 5.0e-02]
 [5.0e-03 0.0e+00 0.0e+00 1.0e+00 1.0e-01]
 [5.0e-03 0.0e+00 0.0e+00 1.0e+00 1.5e-01]
 ...
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.0e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 1.5e-01]
 [2.0e-02 4.9e+01 0.0e+00 1.0e+00 2.0e-01]]


# Checking aux_info keys

In [17]:
import tqdm

recent_experiment_names = [
    "CR_fvt_training_ensemble_max_fvt", 
    "CR_fvt_training_ensemble_max_fvt_HH4b_400",
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_smeared_HH4b_400",
    "base_fvt_training_ensemble",
    "base_fvt_training_ensemble_HH4b_400",
    "base_fvt_training_ensemble_HH4b_800",
    "smeared_fvt_training_ensemble",
    "smeared_fvt_training_ensemble_HH4b_400",
]


for experiment_name in recent_experiment_names:
    print(experiment_name)
    hashes, hparams = TrainingInfo.find(
        {"experiment_name": experiment_name},
        return_hparams=True,
        use_cached_metadata=True
    )
    aux_info_keys = []
    random_idx = np.random.choice(range(len(hashes)), size=100, replace=False)
    for hash_ in tqdm.tqdm([hashes[i] for i in random_idx]):
        tinfo = TrainingInfo.load(hash_)
        aux_info_key = tuple(tinfo.aux_info.keys())
        if aux_info_key not in aux_info_keys:
            aux_info_keys.append(aux_info_key)
    print(aux_info_keys)

CR_fvt_training_ensemble_max_fvt


100%|██████████| 100/100 [00:00<00:00, 1097.82it/s]


[('description', 'step', 'fvt_scores_train_SR', 'fvt_scores_tst_SR')]
CR_fvt_training_ensemble_max_fvt_HH4b_400


100%|██████████| 100/100 [00:00<00:00, 964.35it/s]


[('description', 'step', 'fvt_scores_train_SR', 'fvt_scores_tst_SR')]
CR_fvt_training_ensemble_max_smeared


...

[('description', 'step', 'base_fvt_score')]
smeared_fvt_training_ensemble


100%|██████████| 100/100 [00:06<00:00, 15.06it/s]


[('description', 'step', 'SR_stats_train', 'SR_stats_tst'), ('description', 'step', 'SR_stats_tst', 'SR_stats_train', 'base_fvt_score_train', 'base_fvt_score_tst'), ('description', 'step', 'SR_stats_tst', 'base_fvt_score_train', 'base_fvt_score_tst', 'SR_stats_train')]
smeared_fvt_training_ensemble_HH4b_400


100%|██████████| 100/100 [00:09<00:00, 10.27it/s]

[('description', 'step', 'smeared_fvt_score', 'SR_stats_train', 'SR_stats_tst'), ('description', 'step', 'smeared_fvt_score', 'SR_stats_train', 'SR_stats_tst', 'base_fvt_score_train', 'base_fvt_score_tst')]


In [19]:
A = 1 / (1 + np.exp(np.random.randn(2*10**7)))
B = 1 / (1 + np.exp(np.random.randn(2*10**7)))
%timeit np.log(A / (1 - A)) - np.log(B / (1 - B))
%timeit A - B


182 ms ± 1.27 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
30.7 ms ± 158 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [20]:
%timeit TrainingInfo.load(hash_)

1.94 ms ± 6.59 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
